# Search re-ranking using Gemini embeddings

This notebook demonstrates the use of embeddings to re-rank search results. This walkthrough will focus on the following objectives:

1. Setting up your development environment and API access to use Gemini.
2. Using Gemini's function calling support to access the Wikipedia API.
3. Embedding content via Gemini API.
4. Re-ranking the search results.

This is how you will implement search re-ranking:

1. The user will make a search query.
2. You will use Wikipedia API to return the relevant search results.
3. The search results will be embedded and their relevance will be evaluated by calculating distance metrics like cosine similarity.
4. The most relevant search result will be returned as the final answer.

:::{.callout-note}

The non-source code materials in this notebook are licensed under Creative Commons - Attribution-ShareAlike CC-BY-SA 4.0, [https://creativecommons.org/licenses/by-sa/4.0/legalcode](https://creativecommons.org/licenses/by-sa/4.0/legalcode).

:::


## Setup

### Install the Google GenAI SDK

Install the Google GenAI SDK from [npm](https://www.npmjs.com/package/@google/genai). 

```bash
$ npm install @google/genai
```

### Setup your API key

You can [create](https://aistudio.google.com/app/apikey) your API key using Google AI Studio with a single click.

Remember to treat your API key like a password. Don't accidentally save it in a notebook or source file you later commit to GitHub. In this notebook we will be storing the API key in a `.env` file. You can also set it as an environment variable or use a secret manager. 

Here's how to set it up in a `.env` file:

```bash
$ touch .env
$ echo "GEMINI_API_KEY=<YOUR_API_KEY>" >> .env
```

:::{.callout-tip}

Another option is to set the API key as an environment variable. You can do this in your terminal with the following command:

```bash
$ export GEMINI_API_KEY="<YOUR_API_KEY>"
```
:::

### Load the API key

To load the API key from the `.env` file, we will use the `dotenv` package. This package loads environment variables from a `.env` file into `process.env`. 

```bash
$ npm install dotenv
```

Then, we can load the API key in our code:


In [2]:
const dotenv = require("dotenv") as typeof import("dotenv");

dotenv.config({
  path: "../.env",
});

const GEMINI_API_KEY = process.env.GEMINI_API_KEY ?? "";
if (!GEMINI_API_KEY) {
  throw new Error("GEMINI_API_KEY is not set in the environment variables");
}
console.log("GEMINI_API_KEY is set in the environment variables");


GEMINI_API_KEY is set in the environment variables


:::{.callout-note}
In our particular case the `.env` is is one directory up from the notebook, hence we need to use `../` to go up one directory. If the `.env` file is in the same directory as the notebook, you can omit it altogether. 

```
│
├── .env
└── examples
    └── Search_reranking_using_embeddings.ipynb
```
:::


### Initialize SDK Client

With the new SDK, now you only need to initialize a client with you API key (or OAuth if using [Vertex AI](https://cloud.google.com/vertex-ai)). The model is now set in each call.


In [3]:
const google = require("@google/genai") as typeof import("@google/genai");

const ai = new google.GoogleGenAI({ apiKey: GEMINI_API_KEY });


### Select a model

Now select the model you want to use in this guide, either by selecting one in the list or writing it down. Keep in mind that some models, like the 2.5 ones are thinking models and thus take slightly more time to respond (cf. [thinking notebook](../quickstarts/Get_started_thinking.ipynb) for more details and in particular learn how to switch the thiking off).


In [4]:
const tslab = require("tslab") as typeof import("tslab");

const MODEL_ID = "gemini-2.5-flash-preview-05-20";


## Define tools

As stated earlier, this tutorial uses Gemini's function calling support to access the Wikipedia API. Please refer to the [docs](https://ai.google.dev/docs/function_calling) to learn more about function calling.


### Define the search function

To cater to the search engine needs, you will design this function in the following way:

- For each search query, the search engine will use the `wikipedia.search` method to get relevant topics.
- From the relevant topics, the engine will choose `n_topics(number)` top candidates and will use `gemini-2.5-flash` to extract relevant information from the page.
- The engine will avoid duplicate entries by maintaining a search history.


In [8]:
/* eslint-disable @typescript-eslint/no-unsafe-call, @typescript-eslint/no-unsafe-assignment, @typescript-eslint/no-unsafe-member-access, @typescript-eslint/no-unsafe-argument */

const wikipedia = require("wikipedia") as typeof import("wikipedia");

async function wikipediaSearch(queries: string[]): Promise<string[]> {
  const n_topics = 3;
  const searchHistory = new Set<string>();
  const searchUrl: string[] = [];
  const summaryResults: string[] = [];

  for (const query of queries) {
    console.log(`Searching for "${query}"`);
    // @ts-expect-error search not available
    const search = await wikipedia.search(query, { limit: n_topics });
    const terms = search.results as { ns: string; title: string; pageid: string }[];

    console.log(`Related search terms: [${terms.map((term) => `'${term.title}'`).join(", ")}]`);

    for (const term of terms) {
      if (searchHistory.has(term.title)) continue;
      searchHistory.add(term.title);

      console.log(`Fetching page: ${term.title}`);
      // @ts-expect-error page not available
      const page = await wikipedia.page(term.title);
      searchUrl.push(page.fullurl);
      const content = await page.content();
      console.log(`Information source: ${page.fullurl}`);

      const response = await ai.models.generateContent({
        model: MODEL_ID,
        contents: `
          Extract relevant information
          about user's query: ${query}
          From this source:

          ${content}

          Note: Do not summarize. Only Extract and return the relevant information
        `,
      });
      const urls = [page.fullurl];
      if (response.candidates?.[0]?.citationMetadata) {
        const extraCitations = response.candidates[0].citationMetadata.citations ?? [];
        const extraUrls = extraCitations.map((src) => src.uri).filter((uri) => uri);
        urls.concat(extraUrls);
      }
      summaryResults.push(`${response.text ?? ""}\n\nBased on:\n${urls.join(",\n ")}`);

      await new Promise((resolve) => setTimeout(resolve, 5000));
    }
  }
  console.log("Information Sources:");
  for (const url of searchUrl) {
    console.log(`- ${url}`);
  }
  return summaryResults;
}


In [8]:
const example = await wikipediaSearch(["What are LLMs?"]);


Searching for "What are LLMs?"
Related search terms: ['Large language model', 'Retrieval-augmented generation', 'Prompt injection']
Fetching page: Large language model
Information source: https://en.wikipedia.org/wiki/Large_language_model
Fetching page: Retrieval-augmented generation
Information source: https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Fetching page: Prompt injection
Information source: https://en.wikipedia.org/wiki/Prompt_injection
Information Sources:
- https://en.wikipedia.org/wiki/Large_language_model
- https://en.wikipedia.org/wiki/Retrieval-augmented_generation
- https://en.wikipedia.org/wiki/Prompt_injection


Here is what the search results look like:


In [9]:
for (const e of example) {
  tslab.display.markdown(e);
  tslab.display.markdown("\n---\n");
}


A large language model (LLM) is a language model trained with self-supervised machine learning on a vast amount of text, designed for natural language processing tasks, especially language generation.
The largest and most capable LLMs are generative pretrained transformers (GPTs).
LLMs can be fine-tuned for specific tasks or guided by prompt engineering.
These models acquire predictive power regarding syntax, semantics, and ontologies inherent in human language corpora, but they also inherit inaccuracies and biases present in the data they are trained in.
An LLM is a type of foundation model (large X model) trained on language.

Based on:
https://en.wikipedia.org/wiki/Large_language_model


---


Large language models (LLMs) are:
*   Models that respond to user queries.
*   "Traditional LLMs that rely on static training data."
*   Can be "LLM-based chatbots" that "generate responses."
*   Models that typically have "pre-existing training data" or an "original training set."
*   Concerned with "language semantics."
*   Able to "synthesize an engaging answer tailored to the user."

LLMs' inherent behaviors and limitations:
*   They "can provide incorrect information."
*   They "can generate misinformation" or "hallucinate."
*   They may not "know" or "understand" the context.
*   They "struggle to recognize when they lack sufficient information to provide a reliable response."
*   They "may generate answers even when they should indicate uncertainty."
*   They "lack the ability to assess its own knowledge limitations."
*   They "may extract statements from a source without considering its context, resulting in an incorrect conclusion."
*   When faced with conflicting information, LLMs "may struggle to determine which source is accurate."
*   They "may combine details from multiple sources producing responses that merge outdated and updated information in a misleading manner."

Based on:
https://en.wikipedia.org/wiki/Retrieval-augmented_generation


---


The provided source does not contain a definition or explanation of what Large Language Models (LLMs) are. It primarily discusses prompt injection as a cybersecurity exploit targeting LLMs.

Based on:
https://en.wikipedia.org/wiki/Prompt_injection


---


### Pass the tools to the model

You can pass a list of `FunctionDeclaration` objects in the `tools` config. If you want the SDK itself to call the function (Auto Function Calling), you can pass a `CallableTool` object to the SDK and it will automatically call the function if the method supports it. 


In [11]:
import { CallableTool, FunctionCall, FunctionDeclaration, Part, Tool } from "@google/genai";

const search_wikipedia: FunctionDeclaration = {
  name: "search_wikipedia",
  description: "Search wikipedia for each query and summarize relevant docs.",
  parameters: {
    type: google.Type.OBJECT,
    properties: {
      queries: {
        type: google.Type.ARRAY,
        description: "A list of search queries.",
        items: {
          type: google.Type.STRING,
          description: "A search query",
        },
      },
    },
    required: ["queries"],
  },
  response: {
    type: google.Type.ARRAY,
    description: "Returns relevant extracted information with respect to the query.",
    items: {
      type: google.Type.STRING,
      description: "Extracted relevant information",
    },
  },
};

const search_wikipedia_tool: CallableTool = {
  async callTool(functionCalls: FunctionCall[]): Promise<Part[]> {
    const parts: Part[] = [];
    for (const fc of functionCalls) {
      const args = fc.args as { queries: string[] };
      const results = await wikipediaSearch(args.queries);
      parts.push(
        google.createPartFromFunctionResponse(fc.id ?? "", fc.name ?? "", {
          name: fc.name ?? "",
          response: results,
        })
      );
    }
    return parts;
  },
  async tool(): Promise<Tool> {
    return { functionDeclarations: [search_wikipedia] };
  },
};


## Generate supporting search queries

In order to have multiple supporting search queries to the user's original query, you will ask the model to generate more such queries. This would help the engine to cover the asked question on comprehensive levels.


In [12]:
const userQuery = (query: string): string => `
  You have access to the Wikipedia API which you will be using
  to answer a user's query. Your job is to generate a list of search queries which
  might answer a user's question. Be creative by using various key-phrases from
  the user's query. To generate variety of queries, ask questions which are
  related to  the user's query that might help to find the answer. The more
  queries you generate the better are the odds of you finding the correct answer.
  Here is an example:
  
  user: Tell me about Cricket World cup 2023 winners.
  
  function_call: wikipedia_search(['What is the name of the team that
  won the Cricket World Cup 2023?', 'Who was the captain of the Cricket World Cup
  2023 winning team?', 'Which country hosted the Cricket World Cup 2023?', 'What
  was the venue of the Cricket World Cup 2023 final match?', 'Cricket World cup 2023',
  'Who lifted the Cricket World Cup 2023 trophy?'])
  
  The search function will return a list of article summaries, use these to
  answer the  user's question.
  
  Here is the user's query: ${query}
`;


In order to yield creative and a more random variety of questions, you will set the model's temperature parameter to a value higher. Values can range from [0.0,1.0], inclusive. A value closer to 1.0 will produce responses that are more varied and creative, while a value closer to 0.0 will typically result in more straightforward responses from the model.


## Enable automatic function calling and call the API

Now start a new chat with `automaticFunctionCalling`. With it enabled, the `Chat` will handle the back and forth required to call the function, and return the final response:


In [20]:
const chat = ai.chats.create({
  model: MODEL_ID,
  config: {
    tools: [search_wikipedia_tool],
    temperature: 0.6,
  },
});


In [21]:
async function query(query: string): Promise<string> {
  const response = await chat.sendMessageStream({
    message: userQuery(query),
  });
  let accumulatedResponse = "";
  for await (const part of response) {
    accumulatedResponse += part.text ?? "";
  }
  return accumulatedResponse;
}


In [22]:
// temporarily make console.warn a no-op to avoid warnings in the output (non-text part in GenerateContentResponse caused by accessing .text)
// https://github.com/googleapis/js-genai/blob/d82aba244bdb804b063ef8a983b2916c00b901d2/src/types.ts#L2005
// copy the original console.warn function to restore it later
const warn_fn = console.warn;
// eslint-disable-next-line @typescript-eslint/no-empty-function, no-empty-function
console.warn = function () {};

const query1 = await query("Explain how deep-sea life survives.");


Searching for "How does deep-sea life survive?"
Related search terms: ['Deep sea', 'Deep-sea fish', 'Still Wakes the Deep']
Fetching page: Deep sea
Information source: https://en.wikipedia.org/wiki/Deep_sea
Fetching page: Deep-sea fish
Information source: https://en.wikipedia.org/wiki/Deep-sea_fish
Fetching page: Still Wakes the Deep
Information source: https://en.wikipedia.org/wiki/Still_Wakes_the_Deep
Searching for "Deep sea food sources"
Related search terms: ['Deep-sea fish', 'Deep-sea community', 'Deep sea']
Fetching page: Deep-sea community
Information source: https://en.wikipedia.org/wiki/Deep-sea_community
Searching for "Adaptations of deep-sea organisms to high pressure"
Related search terms: ['Deep-sea fish', 'Deep sea', 'Deep-sea gigantism']
Fetching page: Deep-sea gigantism
Information source: https://en.wikipedia.org/wiki/Deep-sea_gigantism
Searching for "Chemosynthesis in deep sea"
Related search terms: ['Deep sea', 'Deep-sea community', 'Chemosynthesis']
Fetching page: C

In [23]:
tslab.display.markdown(query1);


Deep-sea life has evolved remarkable adaptations to survive in an environment characterized by extreme pressure, perpetual darkness, scarce food, and cold temperatures.

Here's how deep-sea organisms manage to thrive:

1.  **Food Sources:**
    *   **Marine Snow:** The primary food source for most deep-sea organisms is "marine snow," which consists of organic detritus, dead organisms, and waste products that slowly drift down from the sunlit upper layers of the ocean.
    *   **Large Food Falls:** Occasional large food falls, such as whale carcasses, provide substantial, albeit infrequent, feasts for scavengers like amphipods, crabs, and hagfish, creating temporary, localized ecosystems.
    *   **Chemosynthesis:** In specific areas like hydrothermal vents and cold seeps, life doesn't rely on sunlight. Instead, chemosynthetic bacteria form the base of the food chain by converting chemicals (like hydrogen sulfide and methane) from the Earth's interior into organic matter. Many organisms, such as giant tube worms, clams, and mussels, form symbiotic relationships with these bacteria, hosting them internally to gain nutrients.

2.  **Adaptations to High Pressure:**
    *   Deep-sea creatures maintain an internal pressure equal to the external hydrostatic pressure, preventing them from being crushed.
    *   Their cell membranes contain a higher proportion of unsaturated fatty acids to maintain fluidity, as high pressure tends to make membranes rigid.
    *   They have developed unique proteins and enzymes that function optimally under extreme pressure, with structural modifications like increased salt bridges in proteins (e.g., α-actin) to enhance stability.
    *   Many deep-sea fish accumulate osmolytes like Trimethylamine N-oxide (TMAO), which protects proteins from destabilization under high pressure.
    *   Some organisms, like the Mariana hadal snailfish, have evolved open skulls or cartilage-based bone structures that can withstand constant high pressure better than rigid, closed bone structures.

3.  **Adaptations to Lack of Light (Darkness):**
    *   **Sensory Enhancements:** Many deep-sea organisms are blind, relying instead on highly developed senses of touch, smell, and sensitivity to pressure changes to navigate and find food or mates.
    *   **Enhanced Vision:** For those with eyes, they are often exceptionally large and sensitive (up to 100 times more sensitive than human eyes), containing only rod cells and sometimes multiple Rhodopsin genes to detect even the faintest bioluminescent light. Some have tubular eyes that look upwards to spot silhouettes of prey against faint light from above.
    *   **Bioluminescence:** Over 50% of deep-sea fish, shrimp, and squid produce their own light through bioluminescence. This light is used for various purposes:
        *   **Hunting:** Lures (e.g., anglerfish) to attract prey.
        *   **Communication:** Unique patterns to find mates or claim territory.
        *   **Defense:** Distracting predators or using "counter-illumination" (lighting up their bellies to match ambient light from above) for camouflage.

4.  **Physical and Metabolic Adaptations:**
    *   **Buoyancy:** Many deep-sea species have jelly-like flesh, high water content, and reduced skeletal and muscular structures, making them less dense and allowing them to remain suspended with minimal energy expenditure. Some squid use flotation chambers filled with ammonium chloride.
    *   **Slow Metabolism:** Due to scarce food and low temperatures, deep-sea animals generally have very slow metabolisms, allowing them to conserve energy and survive long periods without food.
    *   **Feeding Strategies:** They are often "lie-in-wait" predators with large, extendable mouths, sharp teeth, and expandable stomachs, enabling them to consume prey as large as or even larger than themselves when an opportunity arises. They are typically non-selective feeders.
    *   **Body Structure:** Many have weak, watery muscles and minimal skeletal structures. Some mesopelagic fish undertake daily vertical migrations, moving to shallower waters at night to feed and returning to deeper, safer waters during the day.

5.  **Reproductive Strategies:**
    *   Finding a mate in the vast, dark deep sea is challenging. Adaptations include:
        *   **Bioluminescent courtship displays.**
        *   **Hermaphroditism:** Being both male and female increases the chances of successful reproduction upon encountering another individual.
        *   **Male Parasitism:** In some anglerfish, the tiny male permanently attaches to the female, fusing circulatory systems and becoming a sperm-producing appendage.

These diverse and specialized adaptations allow deep-sea organisms to not only survive but also flourish in one of Earth's most extreme environments.

That looks like it worked. You can go through the chat history to see the details of what was sent and received in the function calls:


In [24]:
import { Content } from "@google/genai";

function printHistory(history: Content[]) {
  for (const content of history) {
    tslab.display.markdown(`### ${content.role ?? ""}:`);
    for (const part of content.parts ?? []) {
      if (part.text) {
        tslab.display.markdown(part.text);
      }
      if (part.functionCall) {
        console.log("Function Call\n", JSON.stringify(part.functionCall, null, 2));
      }
      if (part.functionResponse) {
        console.log("Function Response\n", JSON.stringify(part.functionResponse, null, 2));
      }
    }
    tslab.display.markdown(`\n---\n`);
  }
}


In [25]:
printHistory(chat.getHistory());


### user:


  You have access to the Wikipedia API which you will be using
  to answer a user's query. Your job is to generate a list of search queries which
  might answer a user's question. Be creative by using various key-phrases from
  the user's query. To generate variety of queries, ask questions which are
  related to  the user's query that might help to find the answer. The more
  queries you generate the better are the odds of you finding the correct answer.
  Here is an example:
  
  user: Tell me about Cricket World cup 2023 winners.
  
  function_call: wikipedia_search(['What is the name of the team that
  won the Cricket World Cup 2023?', 'Who was the captain of the Cricket World Cup
  2023 winning team?', 'Which country hosted the Cricket World Cup 2023?', 'What
  was the venue of the Cricket World Cup 2023 final match?', 'Cricket World cup 2023',
  'Who lifted the Cricket World Cup 2023 trophy?'])
  
  The search function will return a list of article summaries, use these to
  answer the  user's question.
  
  Here is the user's query: Explain how deep-sea life survives.



---


### model:

Function Call
 {
  "name": "search_wikipedia",
  "args": {
    "queries": [
      "How does deep-sea life survive?",
      "Deep sea food sources",
      "Adaptations of deep-sea organisms to high pressure",
      "Chemosynthesis in deep sea",
      "Hydrothermal vents deep sea life",
      "Deep sea animal adaptations to lack of light",
      "Deep sea extremophiles",
      "Bioluminescence deep sea creatures",
      "Deep sea fish survival mechanisms",
      "What challenges do deep sea animals face?"
    ]
  }
}


...

 encountering another individual.
        *   **Male Parasitism:** In some anglerfish, the tiny male permanently attaches to the female, fusing circulatory systems and becoming a sperm-producing appendage.

These diverse and specialized adaptations allow deep-sea organisms


---


### model:

 to not only survive but also flourish in one of Earth's most extreme environments.


---


In the chat history you can see all 4 steps:

1. The user sent the query.
2. The model replied with a `FunctionCall` calling the `wikipedia_search` with a number of relevant searches.
3. Because you set `Chat` has auto function calling enabled by default for `sendMessageStream`, it executed the search function and returned the list of article summaries to the model.
4. Following the instructions in the prompt, the model generated a final answer based on those summaries.


## [Optional] Manually execute the function call

If you want to understand what happened behind the scenes, this section executes the `FunctionCall` manually to demonstrate.


In [26]:
const manualChat = ai.chats.create({
  model: MODEL_ID,
  config: {
    tools: [search_wikipedia_tool],
    temperature: 0.6,
    automaticFunctionCalling: {
      disable: true,
    },
  },
});


In [27]:
const manualChatResponse1 = await manualChat.sendMessage({
  message: userQuery("Explain how deep-sea life survives."),
});


Initially the model returns a `FunctionCall`:


In [29]:
console.log(JSON.stringify(manualChatResponse1.functionCalls, null, 2));


[
  {
    "name": "search_wikipedia",
    "args": {
      "queries": [
        "How does deep-sea life survive?",
        "Deep-sea adaptations",
        "Deep-sea food sources",
        "Chemosynthesis deep sea",
        "Hydrothermal vents deep sea life",
        "Deep-sea pressure adaptation",
        "Deep-sea temperature adaptation",
        "Deep-sea bioluminescence",
        "Deep-sea extremophiles"
      ]
    }
  }
]


Call the function with generated arguments to get the results.


In [33]:
const functionCall = manualChatResponse1.functionCalls?.[0];
const { queries } = functionCall?.args as { queries: string[] };
console.log("Function Call Arguments:", JSON.stringify(queries, null, 2));
const summaries = await wikipediaSearch(queries);


Function Call Arguments: [
  "How does deep-sea life survive?",
  "Deep-sea adaptations",
  "Deep-sea food sources",
  "Chemosynthesis deep sea",
  "Hydrothermal vents deep sea life",
  "Deep-sea pressure adaptation",
  "Deep-sea temperature adaptation",
  "Deep-sea bioluminescence",
  "Deep-sea extremophiles"
]
Searching for "How does deep-sea life survive?"
Related search terms: ['Deep sea', 'Deep-sea fish', 'Still Wakes the Deep']
Fetching page: Deep sea
Information source: https://en.wikipedia.org/wiki/Deep_sea
Fetching page: Deep-sea fish
Information source: https://en.wikipedia.org/wiki/Deep-sea_fish
Fetching page: Still Wakes the Deep
Information source: https://en.wikipedia.org/wiki/Still_Wakes_the_Deep
Searching for "Deep-sea adaptations"
Related search terms: ['Deep-sea fish', 'Deep sea', 'Deep-sea gigantism']
Fetching page: Deep-sea gigantism
Information source: https://en.wikipedia.org/wiki/Deep-sea_gigantism
Searching for "Deep-sea food sources"
Related search terms: ['Dee

Now send the `FunctionResponse` to the model.


In [34]:
const functionResponse = google.createPartFromFunctionResponse(functionCall.id ?? "", functionCall.name ?? "", {
  name: functionCall.name ?? "",
  response: summaries,
});

const manualChatResponse2 = await manualChat.sendMessage({
  message: functionResponse,
});


In [35]:
tslab.display.markdown(manualChatResponse2.text ?? "");


Deep-sea life survives in an environment characterized by extreme pressure, perpetual darkness, and scarce food, through a remarkable array of biological and physiological adaptations:

**1. Adapting to Extreme Pressure:**
Deep-sea organisms maintain internal pressure equal to the immense external hydrostatic pressure. Their bodies exhibit several adaptations to prevent collapse:
*   **Protein Structure:** Proteins are modified to function under high pressure, for example, through increased salt bridges in their structure (e.g., in actin) or the presence of osmolytes like Trimethylamine N-oxide (TMAO) that protect proteins.
*   **Cell Membranes:** Cell membranes have a higher proportion of unsaturated fatty acids, which helps maintain their fluidity despite the pressure.
*   **Skeletal and Body Structure:** Many deep-sea fish have gelatinous, watery muscles and minimal skeletal structures (e.g., open skulls in hadal snailfish) to avoid being crushed. Gas-filled swim bladders, common in shallow-water fish, are often absent or filled with fat, as gas would be compressed at such depths.

**2. Surviving in Darkness:**
With no sunlight, deep-sea creatures have evolved unique ways to navigate, find food, and avoid predators:
*   **Sensory Reliance:** They rely heavily on senses other than sight, such as sensitivity to changes in local pressure, smell, and touch, often using long feelers.
*   **Vision:** Those that do have eyes often possess large, sensitive, tubular eyes with only rod cells and an upward field of vision to detect faint light or silhouettes of prey. Some have multiple rhodopsin genes or retroreflectors behind the retina for enhanced low-light vision.
*   **Bioluminescence:** Many organisms produce their own light through bioluminescence. This light is used for:
    *   **Attracting Prey:** Like the anglerfish's glowing lure.
    *   **Communication:** To find mates or signal within their species.
    *   **Camouflage:** Through counter-illumination, where light is emitted from the belly to match the dim light from above, eliminating a shadow.
    *   **Distraction:** Releasing a cloud of luminous material to startle or distract predators.
*   **Coloration:** Many are black or red, as red light wavelengths do not penetrate to the deep sea, making red effectively invisible in the dark.

**3. Overcoming Food Scarcity:**
Food is extremely scarce, so deep-sea life has adapted diverse strategies for energy acquisition:
*   **Marine Snow:** The primary food source for many, consisting of organic detritus, fecal matter, and dead organisms that drift down from the more productive upper layers of the ocean.
*   **Chemosynthesis:** In unique ecosystems around **hydrothermal vents** and **cold seeps**, life thrives without sunlight. Chemosynthetic bacteria form the base of the food chain by converting chemical compounds (like hydrogen sulfide and methane) into organic matter. Many animals, such as tube worms and clams, live in symbiotic relationships with these bacteria, hosting them in their tissues to obtain nutrients.
*   **Whale Falls:** The carcasses of whales that sink to the seafloor provide massive, albeit temporary, food sources, supporting a succession of scavengers and specialized organisms.
*   **Opportunistic Feeding:** Many are "sit-and-wait" predators with slow metabolisms to conserve energy. They often have large, extendable, hinged jaws and sharp teeth to capture and swallow prey larger than themselves when the opportunity arises.
*   **Deep-Sea Gigantism:** Some deep-sea species are significantly larger than their shallow-water relatives. This is thought to be an adaptation to scarce food (larger size improves foraging efficiency), colder temperatures (leading to increased cell size and lifespan), reduced predation pressure, and increased dissolved oxygen concentrations.

**4. Reproduction in a Sparse Environment:**
Finding a mate in the vast, dark deep sea is challenging, leading to adaptations like:
*   **Hermaphroditism:** Many species are hermaphroditic, increasing the chances of successful reproduction when an encounter occurs.
*   **Chemical Signals:** Pheromones are used to attract mates over distances.
*   **Permanent Mating:** In some species, like anglerfish, the male permanently attaches to the female, ensuring a mate is always available for spawning.

**5. Buoyancy and Movement:**
*   **Low-Density Tissues:** Many organisms have jelly-like flesh, high fat content, and reduced skeletal weight to achieve neutral buoyancy without expending much energy.
*   **Efficient Movement:** Body shapes are often adapted for periodic bursts of swimming rather than continuous movement, further conserving energy.

## Re-ranking the search results

Helper function to embed the content:


In [36]:
const EMBEDDING_MODEL_ID = "gemini-embedding-001";

async function embedContent(contents: string[]): Promise<number[][]> {
  const response = await ai.models.embedContent({
    model: EMBEDDING_MODEL_ID,
    contents: contents,
    config: {
      taskType: "semantic_similarity",
    },
  });
  return response.embeddings?.map((embedding) => embedding.values) ?? [];
}


Please refer to the [embeddings guide](https://ai.google.dev/gemini-api/docs/embeddings) for more information on embeddings.

Your next step is to define functions that you can use to calculate similarity scores between two embedding vectors. These scores will help you decide which embedding vector is the most relevant vector to the user's query.

You will now implement cosine similarity as your metric. Here returned embedding vectors will be of unit length and hence their L1 norm will be ~1. Hence, calculating cosine similarity is esentially same as calculating their dot product score.


In [37]:
function dotProduct(a: number[], b: number[]): number {
  return a.reduce((sum, value, index) => sum + value * b[index], 0);
}


### Similarity with user's query

Now it's time to find the most relevant search result returned by the Wikipedia API.

Use Gemini API to get embeddings for user's query and search results.


In [38]:
const searchResults = await embedContent(summaries);
const queryEmbedding = await embedContent(["Explain how deep-sea life survives."]);


Calculate similarity score:


In [42]:
const similarityScores = searchResults.map((result) => dotProduct(result, queryEmbedding[0]));
console.log("Similarity Scores:", similarityScores);


Similarity Scores: [
  0.9117504634217665,
  0.878951087997969,
  0.8209106391648435,
  0.8783796961153858,
  0.8736032494958526,
  0.8513247283724289,
  0.8334251232616245,
  0.789654151444905,
  0.8101736873042428,
  0.8353845714633688
]


Order the summary search results by their similarity scores to the user's query. The highest score will be the most relevant result.

**Users's Input**: Explain how deep-sea life survives.

**Answer**:


In [43]:
const bestCandidateIndex = similarityScores.indexOf(Math.max(...similarityScores));
console.log(`Best Candidate Index: ${bestCandidateIndex} with score ${similarityScores[bestCandidateIndex]}`);
tslab.display.markdown(summaries[bestCandidateIndex] ?? "");


Best Candidate Index: 0 with score 0.9117504634217665


Deep-sea life survives through a variety of adaptations to conditions of low temperatures, darkness, and high pressure, and through diverse feeding methods.

**Feeding and Energy Acquisition:**
*   Organisms survive through feeding methods including scavenging, predation, and filtration.
*   Many organisms survive by feeding on marine snow, which is organic material that has fallen from upper waters.
*   Energy generally comes from organic material (algal particulates, detritus, biological waste, marine snow) drifting down from the photic zone.
*   Food also consists of carcasses derived from the productive zone above.
*   There are many scavengers that feed primarily or entirely upon large food falls, such as whale carcasses.
*   A number of filter feeders feed upon organic particles using tentacles.
*   At hydrothermal vents, some species and communities do not primarily rely upon dissolved organic matter for food but depend on chemosynthesis, for example, through a symbiotic relationship between tube worms and chemosynthetic bacteria. These communities are one of the few ecosystems that do not rely upon sunlight for energy.

**Adaptations to Physical Conditions (Pressure, Buoyancy, Body Structure, Metabolism):**
*   Many deep-sea species have jelly-like flesh consisting mostly of glycosaminoglycans, which provides them with very low density for buoyancy, instead of relying on gas.
*   Deep water squid combine gelatinous tissue with a flotation chamber filled with coelomic fluid made up of ammonium chloride, which is lighter than the surrounding water.
*   Midwater fish are small, usually under 25 centimetres, have slow metabolisms and unspecialized diets, preferring to sit and wait for food to conserve energy.
*   They have elongated bodies with weak, watery muscles and skeletal structures.
*   They often have extendable, hinged jaws with recurved teeth for feeding.
*   Deep-sea fish have adaptations in their proteins, anatomical structures, and metabolic systems to withstand great hydrostatic pressure.
*   They maintain well-regulated metabolic systems despite high pressures.
*   They preserve protein functionality against pressure through physiological and structural adaptations.
*   Some deep-sea fish developed pressure tolerance through changes in the mechanism of their α-actin protein, specifically through substitutions on its active sites (e.g., Q137K, V54A, I67P), leading to significant changes in salt bridge patterns for better stabilization in ATP binding and subunit arrangement. Deep-sea fish have more salt bridges in their actins.
*   Specific osmolytes are abundant in deep-sea fish under high hydrostatic pressure; for certain chondrichthyans, Trimethylamine N-oxide (TMAO) increases with depth, protecting proteins from destabilization by high hydrostatic pressure.
*   Mariana hadal snailfish developed modifications in the Osteocalcin gene (e.g., premature termination) which resulted in open skull and cartilage-based bone formation, as closed skulls and common bone developments of surface vertebrates cannot withstand the extreme hydrostatic pressure.

**Adaptations to Darkness and Reproduction:**
*   Many organisms are hermaphroditic due to sparse distribution and difficulty in finding a partner for breeding in the dark.
*   Fish often have larger than normal, tubular eyes with only rod cells and an upward field of vision to seek out the silhouette of possible prey.
*   Prey fish reduce their silhouettes through lateral compression of the body and counter illumination via bioluminescence (producing light from ventral photophores) to camouflage themselves.
*   Some fish have a retroreflector behind the retina for more sensitive vision in low light.
*   Flashlight fish use a retroreflector plus photophores to detect eyeshine in other fish.

Based on:
https://en.wikipedia.org/wiki/Deep_sea

### Similarity with Hypothetical Document Embeddings (HyDE)

Drawing inspiration from [Gao et al](https://arxiv.org/abs/2212.10496) the objective here is to generate a template answer to the user's query using `gemini-2.5-flash`'s internal knowledge. This hypothetical answer will serve as a baseline to calculate relevance of all the search results.


In [44]:
const res = await ai.models.generateContent({
  model: MODEL_ID,
  contents: `
      Generate a hypothetical answer
      to the user's query by using your own knowledge. Assume that you know everything
      about the said topic. Do not use factual information, instead use placeholders
      to complete your answer. Your answer should feel like it has been written by a human.

      query: "Explain how deep-sea life survives."
    `,
});
tslab.display.markdown(res.text ?? "");


Oh, it's truly astonishing how life manages to thrive in the deepest parts of our planet, isn't it? It's a world of immense pressure, perpetual darkness, and biting cold, yet these incredible organisms have developed some remarkably ingenious solutions to survive, and even flourish.

One of the most immediate challenges is the **overwhelming pressure**. Imagine the weight! Deep-sea life deals with this by having bodies that are fundamentally different from surface creatures. They often lack the air-filled spaces that would collapse under such force. Instead, their internal structures and cellular components are designed to perfectly balance the external force. Many possess specialized **[placeholder for internal stabilizing compounds]** and unique **[placeholder for cellular architecture]** that maintain the integrity of their tissues and organs, preventing them from being crushed or deformed. Their very **[placeholder for bodily consistency]** is often adapted to be in equilibrium with their surroundings.

Then there's the **complete absence of sunlight**. Down there, traditional photosynthesis, which powers most life on the surface, is impossible. So, how do they get energy? This is where it gets truly fascinating. Many deep-sea inhabitants rely on what you might call 'alternative energy pathways.' Instead of the sun, they tap into energy from **[placeholder for geological processes]** or **[placeholder for chemical reactions]** occurring at **[placeholder for specific deep-sea features]**. They often do this with the help of tiny **[placeholder for symbiotic organisms]** that live within or near them, converting these **[placeholder for energy-rich substances]** into usable fuel. Other creatures are masters of efficiency, extracting every last bit of nutrient from the scarce organic 'rain' that drifts down from above, or are adept predators in the dark, some even generating their own **[placeholder for light-like emissions]** for communication or hunting.

The **extreme cold** is another hurdle. Deep-sea organisms manage this by having very efficient **[placeholder for metabolic rates]** that allow them to operate optimally at low temperatures. Some even produce specialized **[placeholder for internal chemical compounds]** that act like a natural antifreeze, preventing ice crystals from forming within their cells.

As for **food scarcity**, it's a truly sparse environment. So, they've adapted by being incredibly efficient and opportunistic. Their **[placeholder for digestive systems]** are often highly specialized to extract maximum nutrients from whatever infrequent meals they find, and many have developed **[placeholder for slow life processes]** or **[placeholder for energy conservation strategies]** to minimize their energy expenditure, allowing them to go for very long periods without food.

Even finding a mate in such a vast, dark expanse is a monumental task. They've developed some truly ingenious solutions, like using powerful **[placeholder for chemical signals]**, unique **[placeholder for light displays]**, or highly sensitive **[placeholder for sensory organs]** to detect each other across great distances.

Ultimately, deep-sea life is a testament to the incredible adaptability of living things. Every aspect of their **[placeholder for biological systems]** – from their molecular makeup to their sensory organs – has been fine-tuned over countless **[placeholder for periods of time]** to not just endure, but to genuinely thrive in one of Earth's most challenging environments. It's a world where **[placeholder for unique biological principles]** seem to be at play, allowing life to persist against all odds.

Use Gemini API to get embeddings for the baseline answer and compare them with search results


In [45]:
const hypotheticalAnswer = await embedContent([res.text ?? ""]);


Calculate similarity scores to rank the search results


In [46]:
const similarityScoresFromHyDE = searchResults.map((result) => dotProduct(result, hypotheticalAnswer[0]));
console.log("Similarity Scores from HyDE:", similarityScoresFromHyDE);


Similarity Scores from HyDE: [
  0.8824987051617097,
  0.8606763507470175,
  0.7755931754943494,
  0.8378522917247573,
  0.8291790160640604,
  0.8156207966301365,
  0.8130383455248943,
  0.7818944367323967,
  0.7959257563521293,
  0.8140033334818754
]


Order the summary search results by their similarity scores to the user's query. The highest score will be the most relevant result.

**Users's Input**: Explain how deep-sea life survives.

**Answer**:


In [47]:
const bestCandidateIndexFromHyDE = similarityScoresFromHyDE.indexOf(Math.max(...similarityScoresFromHyDE));
console.log(
  `Best Candidate Index from HyDE: ${bestCandidateIndexFromHyDE} with score ${similarityScoresFromHyDE[bestCandidateIndexFromHyDE]}`
);
tslab.display.markdown(summaries[bestCandidateIndexFromHyDE] ?? "");


Best Candidate Index from HyDE: 0 with score 0.8824987051617097


Deep-sea life survives through a variety of adaptations to conditions of low temperatures, darkness, and high pressure, and through diverse feeding methods.

**Feeding and Energy Acquisition:**
*   Organisms survive through feeding methods including scavenging, predation, and filtration.
*   Many organisms survive by feeding on marine snow, which is organic material that has fallen from upper waters.
*   Energy generally comes from organic material (algal particulates, detritus, biological waste, marine snow) drifting down from the photic zone.
*   Food also consists of carcasses derived from the productive zone above.
*   There are many scavengers that feed primarily or entirely upon large food falls, such as whale carcasses.
*   A number of filter feeders feed upon organic particles using tentacles.
*   At hydrothermal vents, some species and communities do not primarily rely upon dissolved organic matter for food but depend on chemosynthesis, for example, through a symbiotic relationship between tube worms and chemosynthetic bacteria. These communities are one of the few ecosystems that do not rely upon sunlight for energy.

**Adaptations to Physical Conditions (Pressure, Buoyancy, Body Structure, Metabolism):**
*   Many deep-sea species have jelly-like flesh consisting mostly of glycosaminoglycans, which provides them with very low density for buoyancy, instead of relying on gas.
*   Deep water squid combine gelatinous tissue with a flotation chamber filled with coelomic fluid made up of ammonium chloride, which is lighter than the surrounding water.
*   Midwater fish are small, usually under 25 centimetres, have slow metabolisms and unspecialized diets, preferring to sit and wait for food to conserve energy.
*   They have elongated bodies with weak, watery muscles and skeletal structures.
*   They often have extendable, hinged jaws with recurved teeth for feeding.
*   Deep-sea fish have adaptations in their proteins, anatomical structures, and metabolic systems to withstand great hydrostatic pressure.
*   They maintain well-regulated metabolic systems despite high pressures.
*   They preserve protein functionality against pressure through physiological and structural adaptations.
*   Some deep-sea fish developed pressure tolerance through changes in the mechanism of their α-actin protein, specifically through substitutions on its active sites (e.g., Q137K, V54A, I67P), leading to significant changes in salt bridge patterns for better stabilization in ATP binding and subunit arrangement. Deep-sea fish have more salt bridges in their actins.
*   Specific osmolytes are abundant in deep-sea fish under high hydrostatic pressure; for certain chondrichthyans, Trimethylamine N-oxide (TMAO) increases with depth, protecting proteins from destabilization by high hydrostatic pressure.
*   Mariana hadal snailfish developed modifications in the Osteocalcin gene (e.g., premature termination) which resulted in open skull and cartilage-based bone formation, as closed skulls and common bone developments of surface vertebrates cannot withstand the extreme hydrostatic pressure.

**Adaptations to Darkness and Reproduction:**
*   Many organisms are hermaphroditic due to sparse distribution and difficulty in finding a partner for breeding in the dark.
*   Fish often have larger than normal, tubular eyes with only rod cells and an upward field of vision to seek out the silhouette of possible prey.
*   Prey fish reduce their silhouettes through lateral compression of the body and counter illumination via bioluminescence (producing light from ventral photophores) to camouflage themselves.
*   Some fish have a retroreflector behind the retina for more sensitive vision in low light.
*   Flashlight fish use a retroreflector plus photophores to detect eyeshine in other fish.

Based on:
https://en.wikipedia.org/wiki/Deep_sea

You have now created a search re-ranking engine using embeddings!


## Next steps

I hope you found this example helpful! Check out more examples in the Gemini API cookbook to learn more.
